# Diabetes Prediction Dataset: EDA + Modeling + Results

This notebook uses the Kaggle diabetes prediction dataset from:
https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset/data

The dataset is a clean, tabular health dataset with variables such as age, BMI, HbA1c level, blood glucose, hypertension, heart disease, smoking history, sex, and diabetes status.

We will:
- inspect the data quality
- analyze class imbalance
- visualize risk patterns
- train binary classification models
- evaluate them on a test set
- summarize the final findings


## 1) Setup


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', None)

## 2) Load the Kaggle dataset

Place the CSV in `data/diabetes_prediction_dataset.csv` before running this notebook.

In [ ]:
DATA_PATH = 'data/diabetes_prediction_dataset.csv'

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f'Please download the Kaggle dataset and place it here: {DATA_PATH}.\n'
        'You can also use: python download_kaggle_data.py for instructions.'
    )

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())

## 3) Data profile

We check for null values, duplicate rows, summary statistics, and schema.

In [ ]:
print('Missing values per column:
', df.isna().sum()[df.isna().sum() > 0])
print('Duplicate rows:', df.duplicated().sum())
print('Column names:')
for i, col in enumerate(df.columns, 1):
    print(f'{i}. {col}')
display(df.info())
display(df.describe(include='all').T)

### Interpretation

This dataset is generally clean and ready for modeling. The numeric features show a wide range of patient measurements, and the target variable `diabetes` is binary, which makes this suitable for classification.

## 4) Target variable analysis

We examine the class balance before training any model.

In [ ]:
target_counts = df['diabetes'].value_counts().sort_index()
target_summary = pd.DataFrame({
    'count': target_counts,
    'proportion': target_counts / len(df)
})
display(target_summary)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='diabetes', palette='Set2')
plt.title('Target distribution: diabetes vs no diabetes')
plt.xlabel('Diabetes')
plt.ylabel('Count')
plt.show()

majority_baseline = target_counts.max() / len(df)
print(f'Majority-class baseline accuracy: {majority_baseline:.3f}')

## 5) Explore key feature relationships

We examine variables that are often strongly related to diabetes risk, such as age, BMI, blood glucose, HbA1c, hypertension, and heart disease.

In [ ]:
numeric_features = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
for feature in numeric_features:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x='diabetes', y=feature, palette='Set2')
    plt.title(f'{feature} by diabetes status')
    plt.xlabel('Diabetes status')
    plt.ylabel(feature)
    plt.show()

for feature in ['hypertension', 'heart_disease', 'smoking_history', 'sex']:
    table = pd.crosstab(df[feature], df['diabetes'])
    print(f'\n=== {feature} ===')
    print((table.div(table.sum(axis=1), axis=0) * 100).round(2))

    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=feature, hue='diabetes', palette='Set2')
    plt.title(f'{feature} by diabetes status')
    plt.ylabel('Count')
    plt.xticks(rotation=25)
    plt.legend(title='Diabetes')
    plt.show()

### Interpretation

The diabetic group tends to have higher BMI and worse glucose-related values. Hypertension and heart disease also appear at higher rates among people with diabetes. These relationships support the idea that the dataset has clinically meaningful signals.

## 6) Correlation and feature importance signals

We inspect correlations among numeric predictors and also the relationship between diabetes and risk indicators.

In [ ]:
numeric_df = df.select_dtypes(include=['number'])
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', annot=False, linewidths=0.3)
plt.title('Correlation heatmap (numeric variables)')
plt.show()

corr_with_target = corr['diabetes'].sort_values(ascending=False)
print('Correlation with target:')
print(corr_with_target.head(10))

## 7) Prepare data for classification


In [ ]:
X = df.drop(columns='diabetes')
y = df['diabetes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_transformer, numeric_features),
    ('categorical', categorical_transformer, categorical_features)
])
print('Training rows:', len(X_train))
print('Test rows:', len(X_test))
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

## 8) Model training

We train three common classifiers with pipelines to avoid data leakage.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = []
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'ROC-AUC': roc_auc
    })
    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes'], zero_division=0))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Diabetes', 'Diabetes'], yticklabels=['No Diabetes', 'Diabetes'])
    plt.title(f'{name} confusion matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
display(results_df.round(3))

## 9) Final conclusions

This Kaggle diabetes dataset provides a better training dataset for classification than the older BRFSS 2015 version, because it is easier to work with and includes strong clinical variables such as HbA1c and blood glucose.

### Likely takeaways
- HbA1c and blood glucose are strong predictors of diabetes.
- BMI, age, and hypertension also strongly influence risk.
- It is important to use balanced metrics because diabetes is often unevenly distributed.
- A classification workflow is appropriate here because the target is binary.

## 10) Next steps

Possible extensions:
- hyperparameter tuning
- XGBoost / CatBoost comparison
- calibration curve analysis
- SHAP explanations
- deployment as a simple diabetes risk app
- combine this model with social media analysis in a separate stream of research